# Model Training Experiments
Test different models and parameters

In [ ]:
import pandas as pd
import numpy as np
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
# Load data
df = pd.read_csv('../data/processed/dataset_metadata.csv')
df = df[df['is_valid'] == True].sample(5000, random_state=42)  # Use 5000 for quick testing
print(f"Using {len(df)} images")

In [ ]:
# Load and process images
def load_image(path, size=64):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    return img.flatten() / 255.0

X = []
y_labels = []
for idx, row in df.iterrows():
    img = load_image(row['image_path'])
    if img is not None:
        X.append(img)
        y_labels.append(row['class_label'])

X = np.array(X)
y, labels = pd.factorize(y_labels)
print(f"Loaded {len(X)} images, {len(labels)} classes")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Experiment 1: Different number of trees

In [ ]:
results = {}
for n_trees in [50, 100, 200, 300]:
    print(f"\nTesting {n_trees} trees...")
    model = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    results[n_trees] = acc
    print(f"Accuracy: {acc:.2%}")

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(list(results.keys()), list(results.values()), marker='o')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Trees')
plt.grid(True)
plt.show()

## Experiment 2: Different image sizes

In [ ]:
# Test 32x32 vs 64x64
for size in [32, 64]:
    print(f"\nTesting {size}x{size} images...")
    X_temp = []
    for idx, row in df.iterrows():
        img = load_image(row['image_path'], size=size)
        if img is not None:
            X_temp.append(img)
    
    X_temp = np.array(X_temp)
    X_tr, X_te, y_tr, y_te = train_test_split(X_temp, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"Accuracy: {acc:.2%}")
    print(f"Feature count: {X_temp.shape[1]}")

## Experiment 3: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Train best model
model = RandomForestClassifier(n_estimators=200, max_depth=30, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()